# Hankel Transform of a Sphere Form Factor

This notebook demonstrates how to compute the Hankel transform of the built-in `sphere` form factor using LibHankel.

The sphere form factor is:

$$F(q) = 3 \frac{\sin(qR) - qR\cos(qR)}{(qR)^3}$$

where $R$ is the sphere radius and $q$ is the scattering vector. The Hankel transform ($\nu = 0$) of $F(q)$ gives the correlation function $G(r)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import libhankel

## Set up parameters

The `sphere` form factor takes two parameters: the radius $R$ and a scale factor $\Delta\rho$ (contrast).

In [ ]:
nu = 0                            # Hankel order
params = np.array([10.0, 1.0])    # [radius R, contrast delta_rho]

r_arr = np.linspace(1.0, 25.0, 50)

## Compute the transform with FFTLog

FFTLog is the recommended default strategy. It needs no convergence parameters — pass an empty dict.

In [ ]:
result_fftlog = libhankel.hankel_transform(
    nu, "sphere", r_arr, params, "FFTLog", {}
)

print("G(r) via FFTLog:", result_fftlog)

## Compare with an adaptive strategy

The adaptive `DHT_Key_51` strategy uses per-point quadrature. It accepts `n_eval` and `eps_rel` to control accuracy.

In [ ]:
strategy_params = {"n_eval": 250, "eps_rel": 1e-9}

result_dht = libhankel.hankel_transform(
    nu, "sphere", r_arr, params, "DHT_Key_51", strategy_params
)

print("G(r) via DHT_Key_51:", result_dht)

## Plot results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

fftlog = np.array(result_fftlog)
dht = np.array(result_dht)

axes[0].plot(r_arr, fftlog, label="FFTLog")
axes[0].plot(r_arr, dht, linestyle="--", label="DHT_Key_51")
axes[0].set_xlabel("r")
axes[0].set_ylabel("G(r)")
axes[0].set_title("Hankel transform of sphere form factor")
axes[0].legend()

diff = np.abs(fftlog - dht)
axes[1].semilogy(r_arr, diff)
axes[1].set_xlabel("r")
axes[1].set_ylabel("|FFTLog - DHT_Key_51|")
axes[1].set_title("Absolute difference between strategies")

plt.tight_layout()
plt.show()